### Importing Libraries

In [ ]:
import re
import nltk
import string
import pickle
import imblearn
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from sklearn.linear_model import Ridge
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

## Reading datasets

In [ ]:
fake_dataset= pd.read_csv("fake_News.csv",low_memory=False)
true_dataset= pd.read_csv("true_News.csv",low_memory=False)
half_true=pd.read_csv("half-true.csv",low_memory=False)
mostly_false=pd.read_csv("mostly-false.csv",low_memory=False)
mostly_true=pd.read_csv("mostly-true.csv",low_memory=False)
pants_fire=pd.read_csv("pants-on-fire.csv",low_memory=False)

## Merging the datasets into one

In [ ]:
df = pd.concat([true_dataset,mostly_true,half_true,mostly_false,pants_fire,fake_dataset,]).reset_index(drop = True)

In [ ]:
data=df[['headline','text','label']]

## Shape of data

In [ ]:
data.shape

## Frequency Distribution of label

In [ ]:
%matplotlib inline
carrier_count = df["label"].value_counts()
sns.set(style="darkgrid")
sns.barplot(carrier_count.index, carrier_count.values, alpha=0.9)
plt.title('Frequency Distribution of category')
plt.ylabel('Number of Occurrences', fontsize=12)
plt.xlabel('Carrier', fontsize=12)
plt.show()

## Removing Null Values 

In [ ]:
df=data.dropna( axis=0, how="any", thresh=None, subset=None, inplace=False)

## Checking Null Values

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

## Combining Title and Text 

In [ ]:
df['News Content'] = df['headline'] +' '+ df['text']
df.head()

In [ ]:
df.tail()

## Randomize data

In [ ]:
df=df.sample(frac=1)

In [ ]:
df.head()

In [ ]:
df.tail()

## Selecting Required Columns

In [ ]:
df=df[['News Content','label']]
df.head()

### Cleaning data

In [ ]:
def wordopt(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text) 
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)    
    return text

In [ ]:
df["News Content"] = df["News Content"].apply(wordopt)

In [ ]:
df.head()

## Tokenization

In [ ]:
from nltk.tokenize import RegexpTokenizer
tokenizer = RegexpTokenizer(r'\w+')
df["News Content"] = df["News Content"].apply(tokenizer.tokenize)

In [ ]:
df.head()

## Removing stop words

In [ ]:
stopwords=nltk.corpus.stopwords.words('english')
def remove_stopwords(text):
    text=[word for word in text if word not in stopwords]
    return text

In [ ]:
df['News Content']=df['News Content'].apply(lambda x:remove_stopwords(x))
df.head()

## Lemmatization

In [ ]:
wn= nltk.WordNetLemmatizer()
def lemmatization(text):
    text=[wn.lemmatize(word) for word in text]
    return text

In [ ]:
df['News Content']=df['News Content'].apply(lambda x:lemmatization(x))
df.head()

## Features Extraction

# Count Vectorization 

In [ ]:
vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 2))
X = vectorizer.fit_transform(df['News Content'].astype('str'))
print(vectorizer.get_feature_names())

## Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
labelencoder=LabelEncoder()
df["encoded_label"] = labelencoder.fit_transform(df['label'].astype(str)) 

In [ ]:
df

In [ ]:
df.to_csv("preprocessed.csv")

## Feature selection from the extracted features of unigram

In [ ]:
from sklearn.feature_selection import chi2
from sklearn.feature_selection import SelectKBest


selector = SelectKBest(chi2, k=100)
selector.fit(X, df['encoded_label'])

bigramdata = selector.transform(X)
print(X.shape)

# Dependent and Independent

In [ ]:
X=bigramdata
y=np.array(df['encoded_label'])

# K-Fold

In [ ]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=10)
kf.get_n_splits(X)
print(kf)

In [ ]:
for train_index, test_index in kf.split(X):
    print("TRAIN:", train_index, "TEST:", test_index)
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

# Models Training

## Multinomial NB

In [ ]:
from sklearn.naive_bayes import MultinomialNB
nab=MultinomialNB()
nab= nab.fit(X_train,y_train)
pred=nab.predict(X_test)
print(classification_report(y_test, pred))
nab_accuracy = round((accuracy_score(y_test, pred)*100),2)

print("Accuracy : " , nab_accuracy , " %")

In [ ]:
matrix =confusion_matrix(y_test, pred)
df_cm = pd.DataFrame(matrix, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(max_depth=2,random_state=0)
rf=rf.fit(X_train,np.ravel(y_train))
pred1=rf.predict(X_test)
print(classification_report(y_test, pred1))
rf_accuracy = round((accuracy_score(y_test, pred1)*100),2)
print("Accuracy : " , rf_accuracy , " %")

In [ ]:
matrix1 =confusion_matrix(y_test, pred1)
df_cm = pd.DataFrame(matrix1, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=0) 
dt.fit(X_train,np.ravel(y_train)) 
pred2=dt.predict(X_test)
print(classification_report(y_test, pred2))
print(confusion_matrix(y_test, pred2))


dt_accuracy = round((accuracy_score(y_test, pred2)*100),2)

print("Accuracy : " , dt_accuracy , " %")

In [ ]:
matrix2 =confusion_matrix(y_test, pred2)
df_cm = pd.DataFrame(matrix2, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

## SVM

In [ ]:
from sklearn import svm
svm = svm.LinearSVC() 
svm.fit(X_train,np.ravel(y_train)) 
pred3=svm.predict(X_test)
print(classification_report(y_test, pred3))
print(confusion_matrix(y_test, pred3))


svm_accuracy = round((accuracy_score(y_test, pred3)*100),2)

print("Accuracy : " , svm_accuracy , " %")

In [ ]:
matrix3 =confusion_matrix(y_test, pred3)
df_cm = pd.DataFrame(matrix3, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

In [ ]:
pickle.dump(svm, open('SVM_model.pkl', 'wb'))

# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
lg=LogisticRegression()
lg.fit(X_train,np.ravel(y_train)) 
pred4=lg.predict(X_test)
print(classification_report(y_test, pred4))
print(confusion_matrix(y_test, pred4))


lg_accuracy = round((accuracy_score(y_test, pred4)*100),2)

print("Accuracy : " , lg_accuracy , " %")

In [ ]:
matrix4 =confusion_matrix(y_test, pred4)
df_cm = pd.DataFrame(matrix4, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

In [ ]:
pickle.dump(lg, open('Logistic_Regression_model.pkl', 'wb'))

# Gradient Boosting Classifier

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gbc=GradientBoostingClassifier(n_estimators=100, learning_rate=1.0,max_depth=1, random_state=0)
gbc.fit(X_train, y_train) 
pred5=gbc.predict(X_test)
print(classification_report(y_test, pred5))
print(confusion_matrix(y_test, pred5))


gbc_accuracy = round((accuracy_score(y_test, pred5)*100),2)

print("Accuracy : " , gbc_accuracy , " %")

In [ ]:
matrix5 =confusion_matrix(y_test, pred5)
df_cm = pd.DataFrame(matrix5, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

### Ensemble Techniques

In [ ]:
from imblearn.ensemble import EasyEnsembleClassifier

In [ ]:
easy=EasyEnsembleClassifier()
easy.fit(X_train,y_train)

In [ ]:
pred6=easy.predict(X_test)
print(confusion_matrix(y_test,pred6))
print(accuracy_score(y_test,pred6))
print(classification_report(y_test,pred6))
ensemble_accuracy = round((accuracy_score(y_test, pred6)*100),2)

print("Accuracy : " , ensemble_accuracy , " %")

In [ ]:
matrix6 =confusion_matrix(y_test, pred6)
df_cm = pd.DataFrame(matrix6, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

# Accuracies

In [ ]:
Models = pd.DataFrame({
    'Models': ['MultinomialNB','Random Forest','Decision Tree','SVM','Logistic Regression','Gradient Boosting Classifier','Easy Ensembling Classifier'],
    
    'Accuracies': np.round([nab_accuracy,rf_accuracy,dt_accuracy,svm_accuracy,lg_accuracy,gbc_accuracy,ensemble_accuracy],2)
                     })
print('Rank of models by Accuracies ')
Models.sort_values(by=['Accuracies'],ascending=False)

# Imbalance data Techniques

### Weight changing via Balanced approach

In [ ]:
from sklearn import svm
weight = svm.LinearSVC( class_weight='balanced')
weight.fit(X_train,y_train)

In [ ]:
y_pred0=weight.predict(X_test)
print(confusion_matrix(y_test,y_pred0))
print(accuracy_score(y_test,y_pred0))
print(classification_report(y_test,y_pred0))
Bweight_changing_accuracy = round((accuracy_score(y_test, y_pred0)*100),2)
print("Accuracy : " , Bweight_changing_accuracy , " %")

In [ ]:
matrix7 =confusion_matrix(y_test, y_pred0)
df_cm = pd.DataFrame(matrix7, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

In [ ]:
pickle.dump(weight, open('balanced_weight_model.pkl', 'wb'))

# Manual Weight Change

In [ ]:
Mweight=svm.LinearSVC( class_weight={0:5,1:1,2:1,3:1,4:1,5:5})
Mweight.fit(X_train,y_train)

In [ ]:
y_pred1=Mweight.predict(X_test)
print(confusion_matrix(y_test,y_pred1))
print(accuracy_score(y_test,y_pred1))
print(classification_report(y_test,y_pred1))
Mweight_changing_accuracy = round((accuracy_score(y_test, y_pred1)*100),2)

print("Accuracy : " , Mweight_changing_accuracy , " %")

In [ ]:
matrix8 =confusion_matrix(y_test, y_pred1)
df_cm = pd.DataFrame(matrix8, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

In [ ]:
from collections import Counter
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

### Under Sampling

In [ ]:
from collections import Counter
from imblearn.under_sampling import RandomUnderSampler  
under_sampler = RandomUnderSampler()
X_train_ns,y_train_ns=under_sampler.fit_resample(X_train,y_train)
print("The number of classes before fit {}".format(Counter(y_train)))
print("The number of classes after fit {}".format(Counter(y_train_ns)))

In [ ]:
under=svm.LinearSVC()
under.fit(X_train_ns,y_train_ns)

In [ ]:
y_pred2=under.predict(X_test)
print(confusion_matrix(y_test,y_pred2))
print(accuracy_score(y_test,y_pred2))
print(classification_report(y_test,y_pred2))
underSample_accuracy = round((accuracy_score(y_test, y_pred2)*100),2)

print("Accuracy : " ,underSample_accuracy , " %")

In [ ]:
matrix9 =confusion_matrix(y_test, y_pred2)
df_cm = pd.DataFrame(matrix9, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

### Over Sampling

In [ ]:
os=RandomOverSampler()
X_train_ns,y_train_ns=os.fit_resample(X_train,y_train)
print("The number of classes before fit {}".format(Counter(y_train)))
print("The number of classes after fit {}".format(Counter(y_train_ns)))

In [ ]:
over=svm.LinearSVC()
over.fit(X_train_ns,y_train_ns)

In [ ]:
y_pred3=over.predict(X_test)
print(confusion_matrix(y_test,y_pred3))
print(accuracy_score(y_test,y_pred3))
print(classification_report(y_test,y_pred3))
overSample_accuracy = round((accuracy_score(y_test, y_pred3)*100),2)

print("Accuracy : " , overSample_accuracy , " %")

In [ ]:
matrix10 =confusion_matrix(y_test, y_pred3)
df_cm = pd.DataFrame(matrix10, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

# SmoteTomek

In [ ]:
smote=svm.LinearSVC()
smote.fit(X_train_ns,y_train_ns)

In [ ]:
y_pred8=smote.predict(X_test)
print(confusion_matrix(y_test,y_pred8))
print(accuracy_score(y_test,y_pred8))
print(classification_report(y_test,y_pred8))
smote_accuracy = round((accuracy_score(y_test, y_pred8)*100),2)

print("Accuracy : " , smote_accuracy , " %")

In [ ]:
matrix11 =confusion_matrix(y_test, y_pred8)
df_cm = pd.DataFrame(matrix11, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

### Cross Validation Like KFOLD and Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
svm = svm.LinearSVC()  
grid={'C':10.0 **np.arange(-2,3),'penalty':['l1','l2']}
cv=KFold(n_splits=10,random_state=None,shuffle=False)

In [ ]:
clf=GridSearchCV(svm,grid,cv=cv,n_jobs=-1,scoring='f1_macro')
clf.fit(X_train,y_train)

In [ ]:
y_pred9=clf.predict(X_test)
print(confusion_matrix(y_test,y_pred9))
print(accuracy_score(y_test,y_pred9))
print(classification_report(y_test,y_pred9))
clf_accuracy = round((accuracy_score(y_test, y_pred9)*100),2)
print("Accuracy : " , clf_accuracy , " %")

In [ ]:
matrix12 =confusion_matrix(y_test, y_pred9)
df_cm = pd.DataFrame(matrix12, columns=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'], index=['true', 'mostly_true','half_true','mostly_false','pants_fire','fake_dataset'])
sns.heatmap(df_cm, annot=True,cmap='Blues',fmt='g')

# Accuracies of models using imbalance techniques

In [ ]:
Models = pd.DataFrame({
    'Models': ['Balanced_weight Change','Under Sampling','Over Sampling','Manual_weight Change','SMOTETomek'],
    
    'Accuracies': np.round([Bweight_changing_accuracy,underSample_accuracy,overSample_accuracy,Mweight_changing_accuracy,smote_accuracy],2)
                     })
print('Rank of models by Accuracies ')
Models.sort_values(by=['Accuracies'],ascending=False) 

# Accuracies of models without imbalance techniques

In [ ]:
Models = pd.DataFrame({
    'Models': ['MultinomialNB','Random Forest','Decision Tree','SVM','Logistic Regression','Gradient Boosting Classifier','Easy Ensembling Classifier','SVM Hyperparameter Tunning'],
    
    'Accuracies': np.round([nab_accuracy,rf_accuracy,dt_accuracy,svm_accuracy,lg_accuracy,gbc_accuracy,ensemble_accuracy,clf_accuracy],2)
                     })
print('Rank of models by Accuracies ')
Models.sort_values(by=['Accuracies'],ascending=False)

# Taking user input in the form of url

In [ ]:
url_input = input("\nPlease enter URL of news article here  : ").strip()

In [ ]:
from newspaper import Article
url = url_input
article = Article(url)
article.download()
article.html
article.parse()
title = article.title
text = article.text

In [ ]:
news = title +' '+text
news=[news]
news

In [ ]:
word_vectorizer = CountVectorizer(
    lowercase=True,
    analyzer='word',
    token_pattern=r'\w{1,}',
    stop_words='english',
    ngram_range=(1, 2),
    max_features =100)
unigramdataGet= word_vectorizer.fit_transform(news)
unigramdata = unigramdataGet.toarray()
unigramdata

In [ ]:
model=pickle.load(open('SVM_model.pkl', 'rb'))

In [ ]:
predicted = model.predict(unigramdata)
if(predicted == 0):
    prediction = "true" 
if(predicted == 1):
    prediction = "mostly true"
if(predicted == 2):
    prediction = "half true" 
if(predicted == 3):
    prediction = "mostly false"
if(predicted == 4):
    prediction = "pants fire" 
if(predicted == 5):
    prediction = "false"
print(prediction)